# RAG Using LangChain

## Goal

Use LangChain to package the RAG pieces.

`Document → Splitter → Embeddings → Vector Store → Retriever → Prompt → Model`

Current LangChain docs also present semantic search and RAG as core learning paths.

In [ ]:
%pip install -q -U langchain langchain-openai langchain-text-splitters langchain-chroma chromadb

In [ ]:
import os
from getpass import getpass

if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("OPENAI_API_KEY: ")

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

llm = ChatOpenAI(model="gpt-5-mini", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

## 1. Documents

LangChain's `Document` keeps text and metadata together.

In [ ]:
docs = [
    Document(
        page_content="Python is used for web development, data science, machine learning, automation and AI.",
        metadata={"source":"python.txt"}
    ),
    Document(
        page_content="RAG means Retrieval-Augmented Generation. It retrieves external information and gives it to an LLM as context.",
        metadata={"source":"rag.txt"}
    ),
    Document(
        page_content="Machine learning is a branch of AI. Common types are supervised, unsupervised and reinforcement learning.",
        metadata={"source":"ml.txt"}
    ),
]

## 2. Recursive chunking

Recursive splitting tries larger natural boundaries before smaller ones.

In [ ]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=120,
    chunk_overlap=20
)

chunks = splitter.split_documents(docs)
print("chunks:", len(chunks))

## 3. Vector store + retriever

In [ ]:
store = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name="langchain_rag"
)

retriever = store.as_retriever(search_kwargs={"k":3})

found = retriever.invoke("What is RAG?")
for d in found:
    print(d.metadata, "->", d.page_content)

## 4. Prompt + model

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer only from the context. If missing, say you do not know.\n\nContext:\n{context}"
    ),
    ("human", "{question}")
])

def format_docs(docs):
    return "\n\n".join(
        f"[Source: {d.metadata.get('source')}]\n{d.page_content}"
        for d in docs
    )

def rag(question):
    docs = retriever.invoke(question)
    messages = prompt.invoke({
        "context": format_docs(docs),
        "question": question,
    })
    return llm.invoke(messages).content, docs

answer, docs = rag("What is RAG?")
print(answer)

## 5. Tune retrieval

`k` is how many chunks are returned.

Too small can miss evidence. Too large can add noise.

In [ ]:
for k in [1,2,3]:
    r = store.as_retriever(search_kwargs={"k":k})
    found = r.invoke("What is Python used for?")
    print("k =", k, "sources =", [d.metadata["source"] for d in found])

## What LangChain changes

It gives reusable interfaces around:
- documents
- splitting
- embeddings
- vector stores
- retrievers
- prompts
- models

The architecture stays the same. The code becomes easier to compose.

## Evaluation

Check:
- correct source retrieved?
- relevant chunk near the top?
- answer supported by context?
- answer relevant to the question?

For bigger systems, use a proper evaluation dataset and metrics such as context precision, context recall, faithfulness and answer relevance.